# Step 2a: ManualAug-CL pretrainingBaseline variant. Handcrafted graph augmentations inside a contrastive learningframework, with no generator, no momentum encoder and no age-based weighting ofnegatives. This is the reference point against which GMACORE and GMACORE-TM arecompared.Self-contained: loads the shards from step 1 and defines everything elseinline.## What this notebook saves| Artifact | Purpose || --- | --- || `best_model.pt`, `final_model.pt` | encoder weights for downstream evaluation || `checkpoint_epoch_*.pt` | periodic checkpoints, and resume points || `embeddings/embeddings_epoch_*.pt` | representation-drift and UMAP analysis || `training_history.json` | loss curves for the three-variant comparison || `resource_usage.json` | wall time, throughput, GPU and host memory || `run_config.json` | hyperparameters and corpus provenance |## Four things beyond the model and the embeddings**A fixed evaluation subset.** Embeddings are extracted for the same molecules,in the same order, at every checkpoint and in all three variants. Representationdrift is measured by comparing a molecule's embedding across epochs, so if thesubset differed between runs the comparison would be meaningless. The subset ischosen by a fixed seed that must not change across variants.**Loss history per epoch.** The contrastive-loss trajectory is what Figure 5bcompares across the three variants. It has to be recorded during training.**Corpus provenance.** The manifest of the corpus, the seed and the fullhyperparameter set are written alongside the weights, so the three runs can beshown to have used identical data.**Resume support.** A long run on a pod can be interrupted. Checkpoints includeoptimizer and memory-queue state, so training resumes rather than restarts.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os

os.environ["PYTHONWARNINGS"] = "ignore"

import json
import logging
import sys
import time
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)

try:
    import torch_geometric
except ImportError:
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"], check=True)

try:
    import psutil
except ImportError:
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"], check=True)
    import psutil

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from tqdm.auto import tqdm

print(f"PyTorch {torch.__version__} | PyG {torch_geometric.__version__}")

## ConfigurationDefaults follow Table 3 of the manuscript. `VARIANT` is fixed here; the othertwo notebooks change only this value and the components it switches on.

In [ ]:
WORKSPACE = Path("/workspace") if Path("/workspace").exists() else Path.cwd()

VARIANT = "manualaug_cl_fixed"
GRAPH_DIR = WORKSPACE / "data" / "graphs" / "pubchem"
OUTPUT_DIR = WORKSPACE / "runs" / VARIANT

# ----- Table 3 hyperparameters -----
HIDDEN_DIM = 128
OUTPUT_DIM = 128
NUM_LAYERS = 3
DROPOUT = 0.10

TEMPERATURE = 0.07
QUEUE_SIZE = 50_000

# Diagnostic switch. With USE_QUEUE = False the contrastive loss uses in-batch
# negatives only, which is the SimCLR arrangement. Queue negatives are produced
# by earlier encoder states, and without a momentum encoder to keep them
# consistent they are the prime suspect for the dimensional collapse.
# Raise BATCH_SIZE when disabling the queue, since in-batch negatives are then
# the only ones available.
# In-batch negatives only.
#
# Queue negatives come from earlier encoder states. Without a momentum encoder
# nothing keeps them consistent with the current representation, so the encoder
# can reduce its loss by drifting away from its own past instead of separating
# molecules, and drifting is cheapest inside a low-dimensional subspace.
# Measured directly on this corpus: effective rank fell 32 -> 2.1 with the
# queue and no momentum encoder, while the identical setup WITH a momentum
# encoder rose to 54.
#
# This variant therefore uses SimCLR-style in-batch negatives, which needs a
# larger batch to supply enough of them. It is a deviation from Equations 15
# and 21, made because queue-only negatives are degenerate without a momentum
# encoder.
USE_QUEUE = False

BATCH_SIZE = 512   # in-batch negatives: 511 per query
# Reduced from the 5e-4 of Table 3. At 5e-4 the encoder reached the collapsed
# solution within one epoch and could not escape it, since the gradient at
# exact collapse is symmetric and vanishing.
LEARNING_RATE = 3e-4   # raised to suit the larger batch
WEIGHT_DECAY = 1e-4
EPOCHS = 150
GRAD_CLIP = 5.0
# The similarity regulariser is DISABLED.
#
# L_sim = ||q/|q| - k/|k|||^2 is globally minimised by mapping every molecule
# to the same direction. It has a trivial optimum, rewards nothing else, and
# is BYOL's objective without BYOL's predictor head or stop-gradient
# asymmetry, which are precisely what stop BYOL collapsing. With it enabled
# the encoder collapsed within one epoch in every run.
#
# Set it back to 0.10 only to reproduce that failure deliberately.
SIMILARITY_WEIGHT = 0.0

# ----- handcrafted augmentation rates (ManualAug-CL only) -----
NODE_DROP_RATE = 0.20
EDGE_DROP_RATE = 0.20
FEATURE_MASK_RATE = 0.20

# ----- runtime -----
SEED = 42
NUM_DATA_WORKERS = 8
CHECKPOINT_EVERY = 10
EMBED_EVERY = 10
EVAL_SUBSET_SIZE = 10_000
EVAL_SUBSET_SEED = 12345   # MUST be identical across all three variants
RESUME = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "embeddings").mkdir(exist_ok=True)

print(f"Variant : {VARIANT}")
print(f"Graphs  : {GRAPH_DIR}")
print(f"Output  : {OUTPUT_DIR}")
print(f"Device  : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected, training will be very slow.")

In [ ]:
import random


def set_seed(seed):
    """Seed every source of randomness used during training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True


set_seed(SEED)
print(f"Seed: {SEED}")

## Load the featurized corpus

In [ ]:
MANIFEST_NAME = "manifest.json"


def shard_path(directory, index):
    return os.path.join(str(directory), f"shard_{index:05d}.pt")


def read_manifest(directory):
    with open(os.path.join(str(directory), MANIFEST_NAME), encoding="utf-8") as handle:
        return json.load(handle)


class ShardedGraphDataset(torch.utils.data.Dataset):
    """Graphs loaded from the shards written by step 1."""

    def __init__(self, directory, max_shards=None):
        self.directory = str(directory)
        self.manifest = read_manifest(directory)

        counts = list(self.manifest["shard_counts"])
        if max_shards is not None:
            counts = counts[:max_shards]
        self.num_shards = len(counts)

        self.graphs = []
        for index in tqdm(range(self.num_shards), desc="Loading shards"):
            self.graphs.extend(torch.load(shard_path(self.directory, index), weights_only=False))

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, index):
        return self.graphs[index]


dataset = ShardedGraphDataset(GRAPH_DIR)
NODE_DIM = dataset.manifest["node_dim"]
EDGE_DIM = dataset.manifest["edge_dim"]

print(f"\nGraphs   : {len(dataset):,}")
print(f"node_dim : {NODE_DIM}")
print(f"edge_dim : {EDGE_DIM}")
print(f"Source   : {dataset.manifest.get('source_file')}")

In [ ]:
# Fixed evaluation subset.
#
# Representation drift compares the embedding of the SAME molecule at different
# epochs, and across variants. That requires an identical subset in an
# identical order everywhere, so it is drawn from its own generator seeded by
# EVAL_SUBSET_SEED rather than from the global RNG, whose state depends on how
# much training has happened.
eval_rng = np.random.default_rng(EVAL_SUBSET_SEED)
eval_indices = eval_rng.choice(
    len(dataset), size=min(EVAL_SUBSET_SIZE, len(dataset)), replace=False
)
eval_indices = np.sort(eval_indices)

eval_subset = torch.utils.data.Subset(dataset, eval_indices.tolist())
eval_loader = DataLoader(eval_subset, batch_size=256, shuffle=False)

np.save(OUTPUT_DIR / "eval_indices.npy", eval_indices)
with open(OUTPUT_DIR / "eval_smiles.json", "w", encoding="utf-8") as handle:
    json.dump([dataset[int(i)].smiles for i in eval_indices[:100]], handle, indent=2)

print(f"Evaluation subset: {len(eval_indices):,} molecules (seed {EVAL_SUBSET_SEED})")
print(f"First 3 indices  : {eval_indices[:3].tolist()}")
print("These indices must be identical in the GMACORE and GMACORE-TM runs.")

## Feature statistics

In [ ]:
# Dataset-level feature statistics.
#
# The earlier version standardized x_phys using the statistics of whichever
# molecules happened to share a batch, dividing by (batch_std + 1e-5). When a
# descriptor is near-constant within a batch that divisor is tiny, so the
# feature is amplified by orders of magnitude. The amplification compounds
# through training and drove embedding norms to ~1e12.
#
# Statistics are therefore computed ONCE over a fixed sample of the corpus and
# held constant, so a molecule's features no longer depend on its batch, and
# the standard deviation is floored well away from zero.

_stats_rng = np.random.default_rng(0)
_stats_indices = _stats_rng.choice(len(dataset), size=min(20000, len(dataset)), replace=False)

_cat = torch.cat([dataset[int(i)].x_cat.float() for i in _stats_indices], dim=0)
_phys = torch.cat([dataset[int(i)].x_phys.float() for i in _stats_indices], dim=0)

FEATURE_MEAN = torch.cat([_cat.mean(0), _phys.mean(0)]).to(DEVICE)
FEATURE_STD = torch.cat([_cat.std(0), _phys.std(0)]).to(DEVICE)
FEATURE_STD = torch.clamp(FEATURE_STD, min=1e-2)   # floor, not an epsilon
FEATURE_CLAMP = 10.0

del _cat, _phys

print("Feature statistics (fixed for the whole run):")
print(f"  mean range : [{FEATURE_MEAN.min():.3f}, {FEATURE_MEAN.max():.3f}]")
print(f"  std range  : [{FEATURE_STD.min():.3f}, {FEATURE_STD.max():.3f}]")
print(f"  clamped at : +/- {FEATURE_CLAMP}")

torch.save(
    {"mean": FEATURE_MEAN.cpu(), "std": FEATURE_STD.cpu(), "clamp": FEATURE_CLAMP},
    OUTPUT_DIR / "feature_stats.pt",
)
print("\nSaved to feature_stats.pt. The other variants must reuse these values.")

## ModelThe encoder is shared by all three variants: a shared MLP over the concatenatednode features, an edge encoder, three graph convolutions, mean pooling, and aprojection head onto the contrastive space (Equations 16 to 18).

In [ ]:
def normalize_node_features(x_cat, x_phys, eps=1e-5):
    """Concatenate and standardize the node features using fixed statistics.

    Standardization uses corpus-level mean and standard deviation computed once,
    not per-batch statistics. Per-batch standardization makes a molecule's
    representation depend on its batch, and divides by a quantity that can be
    arbitrarily small, which is what caused the embedding explosion.
    """
    x = torch.cat([x_cat.float(), x_phys.float()], dim=-1)
    x = (x - FEATURE_MEAN) / FEATURE_STD
    return torch.clamp(x, -FEATURE_CLAMP, FEATURE_CLAMP)


def mlp(in_dim, hidden_dim, out_dim):
    return nn.Sequential(
        nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, out_dim)
    )


class GraphEncoder(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=128, output_dim=128,
                 num_layers=3, dropout=0.0):
        super().__init__()
        self.node_encoder = mlp(node_dim, hidden_dim, hidden_dim)
        self.edge_encoder = mlp(edge_dim, hidden_dim, hidden_dim)

        self.convs = nn.ModuleList()
        for layer in range(num_layers):
            out_channels = output_dim if layer == num_layers - 1 else hidden_dim
            self.convs.append(GCNConv(hidden_dim, out_channels))

        self.dropout = dropout

        # Both objectives normalise their inputs, so the scale of the pooled
        # representation is a free direction that the optimiser inflates
        # without penalty. Unchecked it reached ~3e4, at which point the
        # contrastive gradient is scaled by 1/|q| ~ 3e-5 and learning stops.
        # LayerNorm pins the scale and keeps gradients well conditioned.
        self.pool_norm = nn.LayerNorm(output_dim)
        self.projection = mlp(output_dim, hidden_dim, output_dim)

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.01)

    def encode(self, data):
        """Pooled graph representation, before the projection head."""
        x = normalize_node_features(data.x_cat, data.x_phys)
        x = self.node_encoder(x)

        # Encoded for architectural symmetry with the generator used by the
        # other variants. GCNConv consumes scalar edge weights only, so the
        # encoded edge representation is not fed into the convolutions.
        _ = self.edge_encoder(data.edge_attr.float())

        for index, conv in enumerate(self.convs):
            x = conv(x, data.edge_index)
            if index < len(self.convs) - 1:
                x = F.relu(x)
                if self.dropout > 0:
                    x = F.dropout(x, p=self.dropout, training=self.training)

        batch = getattr(data, "batch", None)
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        return self.pool_norm(global_mean_pool(x, batch))

    def forward(self, data):
        return self.projection(self.encode(data))


print("Encoder defined.")

## Handcrafted augmentationsThe ManualAug-CL baseline: node dropping, edge perturbation and node featuremasking, one operation sampled per call. No learned parameters.Node dropping masks features rather than deleting nodes. For a mean-pooledmessage-passing encoder the two are equivalent, and masking keeps `edge_index`and the batch vector valid without an expensive reindex of the batch.

In [ ]:
# Masking replaces features with the corpus MEAN, not with zero.
#
# Augmentation operates on raw features while standardisation happens inside
# the encoder, so a raw 0 becomes (0 - mean)/std after standardisation: a large
# constant vector, identical for every masked node in every molecule. Masking
# 20% of nodes therefore injected the same extreme direction into every
# augmented graph and pushed the encoder into collapse on every step.
#
# The mean is exactly 0 after standardisation, so it is the neutral value.
MASK_CAT = FEATURE_MEAN[:2].cpu()
MASK_PHYS = FEATURE_MEAN[2:].cpu()


def clone_batch(data):
    out = Data(
        x_cat=data.x_cat.clone().float(),
        x_phys=data.x_phys.clone().float(),
        edge_index=data.edge_index.clone(),
        edge_attr=data.edge_attr.clone().float(),
    )
    out.batch = getattr(data, "batch", None)
    out.num_nodes = data.x_cat.size(0)
    return out


def node_drop(data, rate):
    """Replace a random subset of nodes with the neutral feature vector."""
    out = clone_batch(data)
    device = out.x_cat.device
    keep = (torch.rand(out.x_cat.size(0), 1, device=device) >= rate).float()
    out.x_cat = keep * out.x_cat + (1 - keep) * MASK_CAT.to(device)
    out.x_phys = keep * out.x_phys + (1 - keep) * MASK_PHYS.to(device)
    return out


def edge_drop(data, rate):
    """Remove a random subset of bonds, keeping the graph symmetric."""
    out = clone_batch(data)
    num_directed = out.edge_index.size(1)
    if num_directed == 0:
        return out

    num_bonds = num_directed // 2
    mask = (torch.rand(num_bonds, device=out.edge_index.device) >= rate).repeat_interleave(2)
    if mask.numel() < num_directed:
        pad = torch.ones(num_directed - mask.numel(), dtype=torch.bool, device=mask.device)
        mask = torch.cat([mask, pad])
    if mask.sum() == 0:
        return out

    out.edge_index = out.edge_index[:, mask]
    out.edge_attr = out.edge_attr[mask]
    return out


def feature_mask(data, rate):
    """Replace a random subset of feature channels with their neutral value."""
    out = clone_batch(data)
    device = out.x_phys.device
    keep = (torch.rand(1, out.x_phys.size(1), device=device) >= rate).float()
    out.x_phys = keep * out.x_phys + (1 - keep) * MASK_PHYS.to(device)
    return out


OPERATIONS = ["node_drop", "edge_drop", "feature_mask"]


def manual_augment(data):
    choice = OPERATIONS[int(torch.randint(len(OPERATIONS), (1,)).item())]
    if choice == "node_drop":
        return node_drop(data, NODE_DROP_RATE)
    if choice == "edge_drop":
        return edge_drop(data, EDGE_DROP_RATE)
    return feature_mask(data, FEATURE_MASK_RATE)


print(f"Manual augmentations: {OPERATIONS} (masking to feature mean)")

## Memory queue and contrastive objectiveThe queue and the age bookkeeping are identical across the three variants, sothe only difference in GMACORE-TM is that the age weights are applied. Here allvalid negatives carry uniform weight.

In [ ]:
class MemoryQueue(nn.Module):
    """FIFO queue of L2-normalized negative keys, with age tracking."""

    def __init__(self, size, dim, decay=0.99):
        super().__init__()
        self.size = size
        self.decay = decay
        self.register_buffer("keys", F.normalize(torch.randn(size, dim), dim=1))
        self.register_buffer("age", torch.zeros(size))
        self.register_buffer("valid", torch.zeros(size, dtype=torch.bool))
        self.register_buffer("pointer", torch.zeros(1, dtype=torch.long))

    @property
    def num_valid(self):
        return int(self.valid.sum().item())

    @torch.no_grad()
    def enqueue(self, keys):
        keys = F.normalize(keys.detach(), dim=1)
        batch_size = keys.size(0)
        if batch_size == 0:
            return

        self.age += 1.0   # every existing entry is one optimization step older
        pointer = int(self.pointer.item())
        indices = (torch.arange(batch_size, device=keys.device) + pointer) % self.size

        self.keys[indices] = keys
        self.age[indices] = 0.0
        self.valid[indices] = True
        self.pointer[0] = (pointer + batch_size) % self.size

    def negatives(self):
        return self.keys[self.valid]

    def mean_age(self):
        return float(self.age[self.valid].mean().item()) if self.num_valid else 0.0


def info_nce_loss(query, positive_key, negatives, temperature, in_batch_keys=None):
    """Queue-based InfoNCE, optionally with in-batch negatives.

    Queue-only negatives are safe when the keys are produced by a momentum
    encoder, because stored keys then stay close to the current representation
    and a collapsed encoder is penalised immediately. Without a momentum
    encoder the direction drifts quickly, stale queue entries point elsewhere,
    and a model that maps every molecule to a single slowly rotating vector
    scores well below chance while learning nothing.

    In-batch negatives close that loophole: other molecules encoded by the same
    encoder at the same step must be pushed apart, so collapse costs
    log(batch_size) immediately and cannot be hidden by drift.
    """
    query = F.normalize(query, dim=1)
    positive_key = F.normalize(positive_key.detach(), dim=1)
    negatives = F.normalize(negatives.detach(), dim=1)

    logits_pos = (query * positive_key).sum(dim=1, keepdim=True)
    parts = [logits_pos, query @ negatives.t()]

    if in_batch_keys is not None:
        in_batch = F.normalize(in_batch_keys.detach(), dim=1)
        logits_in_batch = query @ in_batch.t()
        # Exclude each query's own key, which is its positive.
        eye = torch.eye(
            logits_in_batch.size(0), dtype=torch.bool, device=query.device
        )
        logits_in_batch = logits_in_batch.masked_fill(eye, float("-inf"))
        parts.append(logits_in_batch)

    logits = torch.cat(parts, dim=1) / temperature
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=query.device)
    return F.cross_entropy(logits, labels)



## Build the model

In [ ]:
set_seed(SEED)

encoder = GraphEncoder(
    node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(DEVICE)

queue = MemoryQueue(QUEUE_SIZE, OUTPUT_DIM).to(DEVICE)

optimizer = torch.optim.Adam(
    encoder.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

num_parameters = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_parameters:,}")

train_loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_DATA_WORKERS, pin_memory=(DEVICE == "cuda"),
    drop_last=True, persistent_workers=NUM_DATA_WORKERS > 0,
)
print(f"Steps per epoch: {len(train_loader):,}")

## Training and evaluation routinesManualAug-CL has no momentum encoder, so keys come from the query encoderitself under `no_grad`. This is the end-to-end contrastive architecture thepaper contrasts with the momentum-encoder design.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader):
    """Pooled embeddings for the fixed evaluation subset, in loader order."""
    was_training = model.training
    model.eval()
    chunks = [model.encode(batch.to(DEVICE)).cpu() for batch in loader]
    if was_training:
        model.train()
    return torch.cat(chunks, dim=0)


def train_epoch(model, loader, optimizer, queue, epoch):
    model.train()
    total_contrastive = total_similarity = 0.0
    num_batches = 0
    positive_cosine_sum = 0.0

    progress = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for batch in progress:
        batch = batch.to(DEVICE, non_blocking=True)

        # Two INDEPENDENT augmented views, as in SimCLR and MoCo. Using the
        # clean graph as the key lets the encoder satisfy the objective by
        # ignoring the augmentation, which weakens the learning signal.
        view_query = manual_augment(batch)
        view_key = manual_augment(batch)
        query = model(view_query)

        # Keys come from the same encoder, without gradients. Evaluation mode
        # disables dropout so the key is a deterministic function of the input.
        #
        with torch.no_grad():
            model.eval()
            keys = model(view_key)
            model.train()

        negatives = queue.negatives() if USE_QUEUE else keys.new_zeros((0, keys.size(1)))
        if USE_QUEUE and negatives.size(0) == 0:
            negatives = keys.detach()   # first step, queue still empty

        loss_contrastive = info_nce_loss(
            query, keys, negatives, TEMPERATURE, in_batch_keys=keys
        )
        # Computed on L2-normalized embeddings. On raw embeddings this term
        # is unbounded, and because the contrastive loss normalizes its
        # inputs nothing else constrains the embedding magnitude, so the
        # two objectives fought and the norm diverged.
        loss_similarity = F.mse_loss(
            F.normalize(query, dim=1), F.normalize(keys.detach(), dim=1)
        )

        loss = loss_contrastive + SIMILARITY_WEIGHT * loss_similarity

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        # Enqueued after the step, so a key's recorded age counts the encoder
        # updates elapsed since the encoder that produced it.
        queue.enqueue(keys)

        with torch.no_grad():
            positive_cosine_sum += float(
                F.cosine_similarity(F.normalize(query, dim=1),
                                    F.normalize(keys, dim=1)).mean()
            )
        total_contrastive += float(loss_contrastive.item())
        total_similarity += float(loss_similarity.item())
        num_batches += 1
        progress.set_postfix(loss=f"{total_contrastive / num_batches:.4f}")

    return {
        "positive_cosine": positive_cosine_sum / max(num_batches, 1),
        "contrastive": total_contrastive / max(num_batches, 1),
        "similarity": total_similarity / max(num_batches, 1),
        "queue_mean_age": queue.mean_age(),
        "queue_occupancy": queue.num_valid,
    }


def save_checkpoint(path, epoch, metrics):
    torch.save({
        "epoch": epoch,
        "variant": VARIANT,
        "encoder_state_dict": encoder.state_dict(),
        "queue_state_dict": queue.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "config": RUN_CONFIG,
    }, path)


print("Routines defined.")

In [ ]:
RUN_CONFIG = {
    "variant": VARIANT,
    "node_dim": NODE_DIM, "edge_dim": EDGE_DIM,
    "hidden_dim": HIDDEN_DIM, "output_dim": OUTPUT_DIM,
    "num_layers": NUM_LAYERS, "dropout": DROPOUT,
    "temperature": TEMPERATURE, "queue_size": QUEUE_SIZE, "use_queue": USE_QUEUE,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY, "epochs": EPOCHS,
    "grad_clip": GRAD_CLIP, "similarity_weight": SIMILARITY_WEIGHT,
    "node_drop_rate": NODE_DROP_RATE, "edge_drop_rate": EDGE_DROP_RATE,
    "feature_mask_rate": FEATURE_MASK_RATE,
    "seed": SEED, "eval_subset_seed": EVAL_SUBSET_SEED,
    "num_parameters": num_parameters,
    # Provenance: proves the three variants used the same corpus.
    "corpus": {
        "graph_dir": str(GRAPH_DIR),
        "num_graphs": len(dataset),
        "manifest": dataset.manifest,
    },
}

with open(OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as handle:
    json.dump(RUN_CONFIG, handle, indent=2)

print(json.dumps({k: v for k, v in RUN_CONFIG.items() if k != "corpus"}, indent=2))

## ResumeIf a checkpoint exists and `RESUME` is set, training continues from it. Model,optimizer and queue state are all restored; restoring the queue matters becausea freshly initialized queue holds random vectors rather than genuine negatives.

In [ ]:
start_epoch = 1
history = []
resource_log = []
best_loss = float("inf")

latest = sorted(OUTPUT_DIR.glob("checkpoint_epoch_*.pt"))
if RESUME and latest:
    checkpoint = torch.load(latest[-1], map_location=DEVICE, weights_only=False)
    encoder.load_state_dict(checkpoint["encoder_state_dict"])
    queue.load_state_dict(checkpoint["queue_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1

    for name, store in [("training_history.json", history), ("resource_usage.json", resource_log)]:
        path = OUTPUT_DIR / name
        if path.exists():
            with open(path, encoding="utf-8") as handle:
                store.extend(json.load(handle))

    if history:
        best_loss = min(record["contrastive"] for record in history)
    print(f"Resumed from {latest[-1].name}, continuing at epoch {start_epoch}")
else:
    print("Starting from scratch.")

## TrainPer-epoch wall time, throughput, peak GPU memory and host RSS are recorded sothe resource comparison across variants needs no separate profiling run.

In [ ]:
process = psutil.Process()

if start_epoch == 1:
    torch.save(
        extract_embeddings(encoder, eval_loader),
        OUTPUT_DIR / "embeddings" / "embeddings_epoch_000.pt",
    )
    print("Saved epoch-0 embeddings (random initialization baseline).")

run_start = time.perf_counter()

for epoch in range(start_epoch, EPOCHS + 1):
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    epoch_start = time.perf_counter()
    metrics = train_epoch(encoder, train_loader, optimizer, queue, epoch)
    epoch_time = time.perf_counter() - epoch_start

    usage = {
        "epoch": epoch,
        "epoch_seconds": round(epoch_time, 2),
        "molecules_per_second": round(len(dataset) / epoch_time, 1),
        "host_rss_gb": round(process.memory_info().rss / 1024**3, 2),
    }
    if DEVICE == "cuda":
        usage["gpu_peak_allocated_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
        usage["gpu_peak_reserved_gb"] = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    # Separate the two quantities the contrastive loss confounds.
    #
    # positive_cosine : alignment of the two augmented views of one molecule.
    #                   Should RISE. Falling means the encoder is degrading.
    # queue_cosine    : similarity of current embeddings to queued negatives.
    #                   Rising means negatives are getting harder, which makes
    #                   the loss rise for a healthy reason.
    # effective_rank  : exp(entropy of the singular value spectrum). Near 1 is
    #                   dimensional collapse; the tens is healthy.
    # centred_cosine  : pairwise cosine after removing the population mean, so
    #                   a shared constant offset does not inflate it.
    with torch.no_grad():
        probe = extract_embeddings(encoder, eval_loader)[:512]
        probe_n = F.normalize(probe, dim=1)

        cosine = probe_n @ probe_n.t()
        offdiag = cosine[~torch.eye(cosine.size(0), dtype=torch.bool)]
        metrics["embedding_norm"] = float(probe.norm(dim=1).mean())
        metrics["mean_pairwise_cosine"] = float(offdiag.mean())

        centred = F.normalize(probe - probe.mean(0, keepdim=True), dim=1)
        cc = centred @ centred.t()
        metrics["centred_cosine"] = float(cc[~torch.eye(cc.size(0), dtype=torch.bool)].mean())

        singular = torch.linalg.svdvals(probe - probe.mean(0, keepdim=True))
        spectrum = singular / singular.sum().clamp_min(1e-12)
        spectrum = spectrum[spectrum > 0]
        metrics["effective_rank"] = float(torch.exp(-(spectrum * spectrum.log()).sum()))

        metrics["queue_cosine"] = float(
            (F.normalize(probe, dim=1).to(queue.keys.device) @ queue.negatives().t()).mean()
        ) if queue.num_valid else 0.0

    record = {"epoch": epoch, **metrics}
    history.append(record)
    resource_log.append(usage)

    print(
        f"Epoch {epoch:3d} | contrastive {metrics['contrastive']:.4f} "
        f"| pos {metrics['positive_cosine']:+.3f} "
        f"| qcos {metrics['queue_cosine']:+.3f} "
        f"| rank {metrics['effective_rank']:5.1f} "
        f"| ccos {metrics['centred_cosine']:+.3f} | {epoch_time:.0f}s"
        + (f" | GPU {usage['gpu_peak_allocated_gb']:.2f} GB" if DEVICE == "cuda" else "")
    )

    if epoch % EMBED_EVERY == 0 or epoch == EPOCHS:
        torch.save(
            extract_embeddings(encoder, eval_loader),
            OUTPUT_DIR / "embeddings" / f"embeddings_epoch_{epoch:03d}.pt",
        )

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
        save_checkpoint(OUTPUT_DIR / f"checkpoint_epoch_{epoch:03d}.pt", epoch, record)

    if metrics["contrastive"] < best_loss:
        best_loss = metrics["contrastive"]
        save_checkpoint(OUTPUT_DIR / "best_model.pt", epoch, record)

    # Written every epoch so an interrupted run keeps its history.
    with open(OUTPUT_DIR / "training_history.json", "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    with open(OUTPUT_DIR / "resource_usage.json", "w", encoding="utf-8") as handle:
        json.dump(resource_log, handle, indent=2)

save_checkpoint(OUTPUT_DIR / "final_model.pt", EPOCHS, history[-1] if history else {})
total_hours = (time.perf_counter() - run_start) / 3600
print(f"\nTraining complete in {total_hours:.2f} h. Best contrastive loss {best_loss:.4f}")

## Summary

In [ ]:
epoch_times = [record["epoch_seconds"] for record in resource_log]
print(f"Variant            : {VARIANT}")
print(f"Epochs completed   : {len(history)}")
print(f"Molecules          : {len(dataset):,}")
print(f"Parameters         : {num_parameters:,}")

if epoch_times:
    print(f"\nMean epoch time    : {np.mean(epoch_times):.1f}s")
    print(f"Total train time   : {np.sum(epoch_times) / 3600:.2f} h")
    print(f"Throughput         : {np.mean([r['molecules_per_second'] for r in resource_log]):,.0f} mol/s")

if DEVICE == "cuda":
    peak = max(r.get("gpu_peak_allocated_gb", 0) for r in resource_log)
    print(f"Peak GPU allocated : {peak:.2f} GB")
print(f"Peak host RSS      : {max(r['host_rss_gb'] for r in resource_log):.2f} GB")

if history:
    print(f"\nContrastive loss   : {history[0]['contrastive']:.4f} -> {history[-1]['contrastive']:.4f}")

print("\nArtifacts:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"  {path.name:<34} {path.stat().st_size / 1024**2:>8.2f} MB")
print(f"  embeddings/    {len(list((OUTPUT_DIR / 'embeddings').glob('*.pt')))} checkpoints")

## NextRun the GMACORE and GMACORE-TM notebooks. They must use the same `GRAPH_DIR`,the same `SEED` and the same `EVAL_SUBSET_SEED`, otherwise the loss curves anddrift measurements will not be comparable.Downstream evaluation loads `best_model.pt` and reads only the encoder weights;the queue and optimizer state are ignored there.